* 시드 결과 확인

In [ ]:
from sqlalchemy import select
from sqlalchemy.orm import Session

from app.db.seed import seed_all
from app.db.seed_data import TEMP_PASSWORD
from app.models.org import User

print("시드 결과 :", seed_all())
print()

with Session(get_engine()) as s:
    for emp_no in ("2019-0412", "2011-0003"):
        user = s.scalars(select(User).where(User.emp_no == emp_no)).one()
        print(emp_no, user.name, " password_hash :",
              user.password_hash[:4] + "...", len(user.password_hash), "자")
    stored_hashes = s.scalars(select(User.password_hash)).all()

print()
print("평문이 그대로 저장된 사람이 있는가 :", TEMP_PASSWORD in stored_hashes)
print("일곱 명이 같은 해시를 쓰는가       :", len(set(stored_hashes)) == 1)


`POST /api/v1/auth/login` 과 `GET /api/v1/auth/me`

`backend/app/schemas/auth.py`

In [ ]:

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field


class LoginIn(BaseModel):
    pass


class UserOut(BaseModel):
    pass


`backend/app/repositories/user_repo.py`

In [ ]:
from __future__ import annotations

from sqlalchemy import select
from sqlalchemy.orm import Session, joinedload

from app.models.org import User


def get_by_emp_no(session: Session, emp_no: str) -> User | None:
    pass


`backend/app/services/auth_service.py`


In [ ]:
from __future__ import annotations

from app.core.exceptions import AuthFailed
from app.core.security import verify_password
from app.db.session import session_scope
from app.models.org import User
from app.repositories import user_repo


def _to_out(user: User) -> dict:
    return {
        "id": user.id,
        "emp_no": user.emp_no,
        "name": user.name,
        "dept": user.dept.name,
        "role": user.role,
        "clearance": user.clearance,
    }


def authenticate(emp_no: str, password: str) -> dict:
    with session_scope() as s:
        user = user_repo.get_by_emp_no(s, emp_no)
        # 사번이 없을 때도, 비밀번호가 틀렸을 때도 같은 예외 · 같은 문구다.
        if user is None or not verify_password(password, user.password_hash):
            raise AuthFailed()
        return _to_out(user)


def get_me(emp_no: str) -> dict:
    with session_scope() as s:
        user = user_repo.get_by_emp_no(s, emp_no)
        if user is None:
            raise AuthFailed()
        return _to_out(user)


* 노트북실행

In [ ]:
from app.services import auth_service

print("로그인 성공 :", auth_service.authenticate("2016-0231", TEMP_PASSWORD))
print()

# 두 실패가 정말 구별되지 않는지 나란히 놓고 본다.
for emp_no, password, label in [
    ("2016-0231", "틀린비밀번호", "비밀번호가 틀렸을 때"),
    ("0000-0000", TEMP_PASSWORD, "사번이 아예 없을 때"),
]:
    try:
        auth_service.authenticate(emp_no, password)
    except AuthFailed as exc:
        print(label, ":", exc.status_code, exc.code, exc.message)


`backend/app/api/v1/auth.py`

In [ ]:
# auth.py
from __future__ import annotations

from typing import Annotated

from fastapi import APIRouter, Header

from app.core.exceptions import AuthFailed
from app.schemas.auth import LoginIn, UserOut
from app.services import auth_service


router = APIRouter(prefix="/auth", tags=["auth"])

@router.post("/login", response_model=UserOut)
def login(body: LoginIn) -> dict:
    pass


@router.get("/me", response_model=UserOut)
def me(x_emp_no: Annotated[str | None, Header()] = None) -> dict:
    pass


`backend/app/main.py`


In [ ]:
...

app = FastAPI(title="사내 규정 에이전트", version="0.1.0", lifespan=lifespan)

# 추가 
app.include_router(auth.router, prefix="/api/v1")       

app.include_router(documents.router, prefix="/api/v1")

...

In [ ]:
# ── 어제까지의 테스트와 오늘 더한 넷을 함께 돌린다 ──
# 터미널에서는 그냥 `pytest -q` 다. 노트북에서는 결과만 받아 오려고 subprocess 로 부른다.
import subprocess

# --color=no  결과 문자열에 색깔 제어문자가 섞이지 않게 한다
# -p no:cacheprovider  .pytest_cache 폴더를 만들지 않게 한다 (레포에 남길 것이 아니다)
done = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no",
                       "-p", "no:cacheprovider"],
                      cwd=ROOT, capture_output=True, text=True)
summary = done.stdout.strip().splitlines()[-1]
print("pytest 요약      :", summary)
print("통과한 테스트 수 :", summary.split()[0])


`frontend/core/session.py`


In [ ]:
from __future__ import annotations

import streamlit as st

DEFAULTS: dict = {
    "page": "login",
    "user": None,
    "f_dept": "전체",
    "f_level": "전체",
    "f_status": "전체",
    "f_q": "",
}


def init_state() -> None:
    for key, value in DEFAULTS.items():
        st.session_state.setdefault(key, value)


def current_user() -> dict | None:
    pass
    


def is_authenticated() -> bool:
    pass


def login(user: dict) -> None:
    pass


def logout() -> None:
    pass


def emp_no() -> str | None:
    pass


`frontend/core/router.py`


In [ ]:
from __future__ import annotations

import streamlit as st


def current_page() -> str:
    pass


def go(page: str) -> None:
    pass


`frontend/views/login.py`

In [ ]:
from __future__ import annotations

import streamlit as st

from core import session


def render() -> None:
    st.title("사내 업무 에이전트")
    st.write("사번과 비밀번호로 로그인하세요. 계정은 강사가 나눠 준 목록에 있습니다.")

    st.text_input("사번", key="login_emp_no", placeholder="예) 2016-0231")
    st.text_input("비밀번호", type="password", key="login_pw")

    if not st.button("로그인"):
        return

    try:
        from core import api_client
    except ImportError:
        st.info("백엔드 호출 통로(core/api_client.py)는 02번에서 붙입니다.")
        return

    try:
        user = api_client.login(
            st.session_state["login_emp_no"],
            st.session_state["login_pw"],
        )
    except api_client.ApiError as exc:
        st.error(str(exc))
        return

    session.login(user)

    st.rerun()


`frontend/app.py`

In [ ]:
from __future__ import annotations

import html
import pathlib
import sys

import streamlit as st

sys.path.insert(0, str(pathlib.Path(__file__).parent))

from core import router, session  
from ui.theme import inject_css  
from views import login as login_view  

st.set_page_config(
    page_title="사내 업무 에이전트",
    layout="wide",
    initial_sidebar_state="expanded",
)

inject_css()

NAV: list[tuple[str, str | None]] = [
    ("AI 업무 도우미", None),      
    ("문서 관리", "documents"),    
    ("승인함", None),              
    ("운영 대시보드", None),       
]


def render_sidebar() -> None:
    st.sidebar.markdown(
        '<div class="ag-brand"><div class="ag-brand-name">사내 업무 에이전트</div></div>',
        unsafe_allow_html=True,
    )

    # 추가 
    user = session.current_user()
    if user is not None:
        st.sidebar.markdown(
            '<div class="ag-user"><div>'
            f'<div class="ag-user-name">{html.escape(user["name"])}</div>'
            f'<div class="ag-user-role">{html.escape(user["dept"])}</div>'
            '</div></div>',
            unsafe_allow_html=True,
        )
        if st.sidebar.button("로그아웃", key="nav_logout"):
            session.logout()
            st.rerun()      

    for label, page_key in NAV:
        if st.sidebar.button(label, key=f"nav_{label}"):
            if page_key is None:
                st.sidebar.info("아직 만들지 않은 화면입니다.")
            else:
                st.session_state["page"] = page_key

# 수정 
def main() -> None:
    session.init_state()

    if not session.is_authenticated():
        login_view.render()
        return

    render_sidebar()

    page = router.current_page()
    if page == "documents":
        st.info("문서 목록 화면은 02번에서 만듭니다.")
    else:
        st.info("아직 만들지 않은 화면입니다.")


main()



```bash
# 터미널 1 — 백엔드
uvicorn app.main:app --app-dir backend --reload --port 8000

# 터미널 2 — 화면
streamlit run frontend/app.py
```